# Converting a Deterministic <span style="font-variant:small-caps;">Dfa</span> into a Regular Expression

In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

The function `regexpSum` takes a set $S = \{ r_1, \cdots, r_n \}$ of regular expressions
as its argument.  It returns the regular expression 
$$ r_1 + \cdots + r_n. $$
The `RegExp` union stores alternatives, concatenations and repetitions in distinct cases.

An empty sum denotes the empty language; a singleton sum is its sole expression.

In [ ]:
let regexpSum expressions : RegExp =
    match Seq.toList expressions with
    | [] -> Empty
    | first :: rest -> List.fold (fun result next -> Alt (result, next)) first rest

The function `rpq` assumes there is some <span style="font-variant:small-caps;">Dfa</span>
$$ F = \langle \texttt{States}, \Sigma, \delta, \texttt{q0}, \texttt{Accepting} \rangle $$
given and takes five arguments:
- `p1` and `p2` are states of the <span style="font-variant:small-caps;">Dfa</span> $F$,
- $\Sigma$ is the alphabet of the <span style="font-variant:small-caps;">Dfa</span>,
- $\delta$ is the transition function of the <span style="font-variant:small-caps;">Dfa</span> $F$, and
- `Allowed` is a subset of the set `States`.  On recursive calls, `Allowed` is a list of states.

The function `rpq` computes a regular expression that describes those strings that take the 
<span style="font-variant:small-caps;">Dfa</span> $F$ from the state `p1` to state `p2`.
When $F$ switches states from `p1` to `p2` only states in the set `Allowed` may be visited in-between the states `p1` and `p2`.

The function is defined by recursion on the set `Allowed`.  There are two cases
- $\texttt{Allowed} = \{\}$.  
  Define `AllChars`as the set of all characters that when read by $F$ in the state `p1` cause $F$ to enter the state `p2`:
  $$ \texttt{AllChars} = \{ c \in \Sigma \mid \delta(p_1, c) = p_2 \} $$
  Then we need a further case distinction:
  - $p_1 = p_2$: In this case we have:
    $$ \texttt{rpq}(p_1, p_2, \{\}) := \sum\limits_{c\in\texttt{AllChars}} c \quad + \varepsilon$$
    If $\texttt{AllChars} = \{\}$ the sum $\sum\limits_{c\in\texttt{AllChars}} c$ is to be interpreted as the
    regular expression $\emptyset$ that denotes the empty language. 
    
    Otherwise, if $\texttt{AllChars} = \{c_1,\cdots,c_n\}$ we have
    $\sum\limits_{c\in\texttt{AllChars}} c \quad = c_1 + \cdots + c_n$.
  - $p_1 \not= p_2$: In this case we have:
    $$ \texttt{rpq}(p_1, p_2, \{\}) := \sum\limits_{c\in\texttt{AllChars}} c \quad$$
- $\texttt{Allowed} = \{ q \} \cup \texttt{RestAllowed}$.  In this case we recursively define the following variables:
  1. $\texttt{rp1p2} := \texttt{rpq}(p_1, p_2, \Sigma, \delta, \texttt{RestAllowed})$,
  2. $\texttt{rp1q } := \texttt{rpq}(p_1, q, \Sigma, \delta, \texttt{RestAllowed})$,
  3. $\texttt{rqq }\texttt{ } := \texttt{rpq}(q, q, \Sigma, \delta, \texttt{RestAllowed})$,
  4. $\texttt{rqp2 } := \texttt{rpq}(q, p_2, \Sigma, \delta, \texttt{RestAllowed})$.

  Then we can define:
  $$ \texttt{rpq}(p_1, p_2, \texttt{Allowed}) := \texttt{rp1p2} + \texttt{rp1q} \cdot \texttt{rqq}^* \cdot \texttt{rqp2} $$
  This formula can be understood as follows:  If a string $w$ is read in state $p_1$ and reading this string takes the 
  <span style="font-variant:small-caps;">Dfa</span> $F$ from the state $p_1$ to the state $p_2$ while only visiting states from the set 
  `Allowed` in-between, then there are two cases:
  - Reading $w$ does not visit the state $q$ in-between.  Hence the string $w$ can be described by the regular expression
    `rp1p2`.
  - The string $w$ can be written as $w = t u_1 \cdots u_n v$ where:
    - reading $t$ in the state $p_1$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ into the state $q$,
    - for all $i \in \{1,\cdots,n\}$ reading $v_i$ in the state $q$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ from $q$ to $q$, and
    - reading $v$ in the state $q$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ into the state $p_2$.

In [ ]:
let rec rpq p1 p2 alphabet (delta: Map<'state * char, 'state>) allowed : RegExp =
    match allowed with
    | [] ->
        let letters =
            alphabet
            |> Seq.filter (fun c -> Map.tryFind (p1, c) delta = Some p2)
            |> Seq.map Literal
            |> regexpSum
        if p1 = p2 then Alt (Epsilon, letters) else letters
    | q :: rest ->
        let rp1p2 = rpq p1 p2 alphabet delta rest
        let rp1q = rpq p1 q alphabet delta rest
        let rqq = rpq q q alphabet delta rest
        let rqp2 = rpq q p2 alphabet delta rest
        Alt (rp1p2, Concat (Concat (rp1q, Star rqq), rqp2))

The function `dfa2regexp` takes a deterministic <span style="font-variant:small-caps;">Dfa</span> $F$ and computes a regular expression $r$ that describes the same language as $F$, i.e. we have
$$ L(A) = L(r). $$

In [ ]:
let dfa2regexp (dfa: Dfa<'state>) : RegExp =
    dfa.Accepting
    |> Seq.map (fun final -> rpq dfa.Start final dfa.Alphabet dfa.Transitions (Set.toList dfa.States))
    |> regexpSum

`regexp2string` renders the result using the lecture's notation. Concatenation is written by juxtaposition, so the result can be parsed again by `RegExpParser.parse`. This direct recursive construction has exponential cost; it follows the proof rather than optimizing away its intermediate expressions.

In [ ]:
let rec regexp2string expression : string =
    match expression with
    | Empty -> "∅"
    | Epsilon -> "𝜀"
    | Literal c -> string c
    | Concat (left, (Concat _ as right)) -> regexp2string left + "(" + regexp2string right + ")"
    | Concat (left, right) -> regexp2string left + regexp2string right
    | Alt (left, right) -> "(" + regexp2string left + "+" + regexp2string right + ")"
    | Star (Literal _ as r) | Star (Epsilon as r) | Star (Empty as r) -> regexp2string r + "*"
    | Star r -> "(" + regexp2string r + ")*"

The notebook `06-Test-DFA-2-RegExp.ipynb` provides a test for the function `dfa_2_regexp`.